# LPQuant Baseline Suite / 基准实验套件

这个 notebook 是当前 LP interval research workflow 的默认入口。

整体流程尽量标准化，方便后面持续做正式研究：

1. 运行 baseline case matrix
2. 查看 pair-level comparison table
3. 用标准图表做横向比较
4. 把整次实验保存成 timestamped experiment bundle
5. 选一个单独 case 做 deep dive


In [ ]:
import pandas as pd

from app.research import (
    BASELINE_CASES,
    build_pair_comparison_table,
    create_experiment_dir,
    plot_study_candidate_frontier,
    plot_summary_metric_bars,
    plot_summary_metric_heatmap,
    rename_for_display,
    run_benchmark_suite,
    run_study,
    save_benchmark_suite,
    save_study_artifacts,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

len(BASELINE_CASES), [case.name for case in BASELINE_CASES[:4]]


## 运行 baseline suite

如果某个 pair 拿不到数据，suite 会继续往下跑，并把失败记录放进 `suite.failures`。


In [ ]:
suite = run_benchmark_suite(BASELINE_CASES)
rename_for_display(suite.summary)


In [ ]:
rename_for_display(suite.failures)


## Pair comparison table

这张表用来快速比较不同 pair、不同 scenario 下的 top-ranked interval。


In [ ]:
comparison_table = build_pair_comparison_table(suite.summary)
comparison_table


## 标准可视化

这些图建议保持固定风格，便于之后做 run-over-run 对比。


In [ ]:
plot_summary_metric_heatmap(
    suite.summary,
    "score",
    title="不同 pair / scenario 的综合得分 heatmap",
)


In [ ]:
plot_summary_metric_heatmap(
    suite.summary,
    "mean_fee_proxy_bps",
    title="不同 pair / scenario 的 fee proxy heatmap",
)


In [ ]:
plot_summary_metric_bars(
    suite.summary,
    "no_exit_rate",
    title="不同 pair / scenario 的 no-exit rate",
)


In [ ]:
plot_summary_metric_bars(
    suite.summary,
    "downside_breach_rate",
    title="不同 pair / scenario 的 downside breach rate",
)


## 保存 experiment bundle

这一段会把整次 benchmark run 保存成一个带 timestamp 的 bundle，后面可以直接做 run history 对比。


In [ ]:
run_dir = create_experiment_dir("research_runs", "baseline_suite")
saved_paths = save_benchmark_suite(
    suite,
    run_dir,
    label="baseline_suite",
    notes="Baseline benchmark matrix，从 notebook 入口直接运行。",
    tags=["baseline", "notebook", "pair-comparison"],
)
saved_paths


## 单个 case deep dive

这里选一个 case 重跑，并查看 candidate frontier、相似历史窗口，以及 focused pair comparison。


In [ ]:
selected_case = next(case for case in BASELINE_CASES if case.name == "sui_usdc_4h_neutral_balanced")
study = run_study(selected_case.request)
rename_for_display(study.rankings.head(20))


In [ ]:
plot_study_candidate_frontier(
    study.rankings.head(20),
    x="mean_in_range_pct",
    y="mean_fee_proxy_bps",
    title=f"Candidate frontier: {selected_case.name}",
)


In [ ]:
rename_for_display(study.similar_windows[[
    "similarity_rank",
    "entry_time",
    "close",
    "trend_pct",
    "realized_vol_pct",
    "distance_to_sma_pct",
    "drawdown_pct",
    "rsi",
    "distance",
]].head(15))


In [ ]:
pair_focus = "SUI/USDC"
build_pair_comparison_table(suite.summary, pair=pair_focus)


In [ ]:
deep_dive_paths = save_study_artifacts(
    study,
    run_dir / "pair_deep_dives" / selected_case.name,
    label=selected_case.name,
    notes="Single-pair deep dive，来自 baseline notebook。",
    tags=["deep-dive", pair_focus.lower().replace("/", "_")],
)
deep_dive_paths
